# 01 · SE(3)、标定与相机投影

## 你要回答的问题

自动驾驶中的 camera、LiDAR、radar 并不是天然处在同一个坐标系。模型输入之前，必须先处理坐标变换、外参、内参和时间对齐。本 notebook 用合成点云构造一个最小但可解释的几何实验。

核心关系：

\[
x_b = T_{b\leftarrow a}x_a,\qquad
u \sim K[R\mid t]x_b
\]

其中 (T\in SE(3)) 是刚体变换，(K) 是相机内参矩阵。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from time import perf_counter

np.set_printoptions(precision=3, suppress=True)
rng = np.random.default_rng(7)

def make_T(yaw_deg=0.0, translation=(0.0, 0.0, 0.0)):
    """Create a planar-yaw SE(3) transform."""
    yaw = np.deg2rad(yaw_deg)
    c, s = np.cos(yaw), np.sin(yaw)
    T = np.eye(4)
    T[:3, :3] = [[c, -s, 0.0], [s, c, 0.0], [0.0, 0.0, 1.0]]
    T[:3, 3] = np.asarray(translation, dtype=float)
    return T

def transform(T, points_xyz):
    points_xyz = np.asarray(points_xyz)
    ones = np.ones((len(points_xyz), 1))
    return (T @ np.concatenate([points_xyz, ones], axis=1).T).T[:, :3]

def project(K, points_camera):
    z = points_camera[:, 2]
    valid = z > 1e-6
    uv = np.full((len(points_camera), 2), np.nan)
    uv[valid] = (K @ points_camera[valid].T).T[:, :2] / z[valid, None]
    return uv, valid

K = np.array([[720.0, 0.0, 640.0], [0.0, 720.0, 360.0], [0.0, 0.0, 1.0]])
K

## 1. 建立 ego → camera 变换

本实验采用一个常见的教学约定：ego 坐标系 (x\) 向前、(y\) 向左、(z\) 向上；camera 坐标系 (z\) 向前、(x\) 向右、(y\) 向下。真实项目中必须以数据集或车辆平台的坐标约定为准，不能依赖记忆。

In [ ]:
R_ce = np.array([
    [0.0, -1.0, 0.0],  # ego y-left -> camera x-right
    [0.0, 0.0, -1.0], # ego z-up -> camera y-down
    [1.0, 0.0, 0.0],  # ego x-forward -> camera z-forward
])
camera_origin_in_ego = np.array([1.3, 0.0, 1.4])
T_c_from_e = np.eye(4)
T_c_from_e[:3, :3] = R_ce
T_c_from_e[:3, 3] = -R_ce @ camera_origin_in_ego

print('T_c_from_e =\n', T_c_from_e)
print('orthogonality error =', np.linalg.norm(R_ce @ R_ce.T - np.eye(3)))

In [ ]:
def make_cube(center=(12.0, 0.0, 0.8), size=(2.0, 1.8, 1.6), n=6):
    cx, cy, cz = center
    sx, sy, sz = np.asarray(size) / 2
    xs = np.linspace(cx - sx, cx + sx, n)
    ys = np.linspace(cy - sy, cy + sy, n)
    zs = np.linspace(cz - sz, cz + sz, n)
    faces = []
    for x in [xs[0], xs[-1]]:
        faces.extend([[x, y, z] for y in ys for z in zs])
    for y in [ys[0], ys[-1]]:
        faces.extend([[x, y, z] for x in xs for z in zs])
    for z in [zs[0], zs[-1]]:
        faces.extend([[x, y, z] for x in xs for y in ys])
    return np.unique(np.asarray(faces), axis=0)

ego_points = make_cube()
camera_points = transform(T_c_from_e, ego_points)
uv, valid = project(K, camera_points)
print('ego points:', ego_points.shape, 'valid projections:', valid.sum())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(ego_points[:, 0], ego_points[:, 1], c=ego_points[:, 2], s=12, cmap='viridis')
axes[0].set_title('Ego frame: top view')
axes[0].set_xlabel('x forward (m)')
axes[0].set_ylabel('y left (m)')
axes[0].axis('equal')
axes[1].scatter(uv[valid, 0], uv[valid, 1], c=camera_points[valid, 2], s=12, cmap='plasma')
axes[1].invert_yaxis()
axes[1].set_xlim(0, 1280)
axes[1].set_ylim(720, 0)
axes[1].set_title('Camera image plane')
axes[1].set_xlabel('u (pixel)')
axes[1].set_ylabel('v (pixel)')
plt.tight_layout()

## 2. 习题：外参误差如何变成检测/融合误差？

下面的实验注入 yaw 外参误差，并计算投影点的平均像素位移。请先预测曲线形状，再运行代码。

**TODO**

1. 把误差从 yaw 扩展到平移误差；
2. 比较近处目标和远处目标对同样角度误差的敏感性；
3. 解释为什么一个小的标定误差可能在 BEV 或远距离目标上变成明显的空间偏差。

In [ ]:
yaw_errors = np.linspace(-2.0, 2.0, 41)
pixel_shift = []
uv_ref, ref_valid = project(K, transform(T_c_from_e, ego_points))

for err in yaw_errors:
    T_err = make_T(err) @ T_c_from_e
    uv_err, err_valid = project(K, transform(T_err, ego_points))
    both = ref_valid & err_valid
    pixel_shift.append(np.nanmean(np.linalg.norm(uv_err[both] - uv_ref[both], axis=1)))

plt.figure(figsize=(7, 4))
plt.plot(yaw_errors, pixel_shift, marker='o', ms=3)
plt.xlabel('Injected yaw error (deg)')
plt.ylabel('Mean pixel displacement')
plt.title('Calibration error → projection error')
plt.grid(alpha=.3)
plt.show()

In [ ]:
try:
    from ipywidgets import interact, FloatSlider
    from IPython.display import clear_output

    def inspect_projection(yaw_error=0.0, tx_error=0.0):
        clear_output(wait=True)
        T_err = make_T(yaw_error, (tx_error, 0.0, 0.0)) @ T_c_from_e
        uv_err, valid_err = project(K, transform(T_err, ego_points))
        both = valid & valid_err
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.scatter(uv[valid, 0], uv[valid, 1], s=14, label='reference')
        ax.scatter(uv_err[valid_err, 0], uv_err[valid_err, 1], s=14, label='perturbed')
        ax.invert_yaxis(); ax.set_xlim(0, 1280); ax.set_ylim(720, 0)
        ax.set_title(f'mean pixel shift = {np.mean(np.linalg.norm(uv_err[both] - uv[both], axis=1)):.2f}')
        ax.legend(); plt.show()

    interact(inspect_projection,
             yaw_error=FloatSlider(min=-2, max=2, step=.1, value=0),
             tx_error=FloatSlider(min=-.5, max=.5, step=.02, value=0));
except ImportError:
    print('Optional interactive controls require: pip install ipywidgets')

## 完成标准

- 能解释 (T_{c\leftarrow e}) 的旋转和平移含义；
- 能说明内参误差、外参误差和时间错位分别影响什么；
- 能把实验结果写成“误差来源 → 空间偏差 → 模型后果”的链条，而不是只贴图。